# VQCC v16 - best-of public portfolio, every member re-verified

Per task, the cheapest circuit that passes the unmodified official verifier (local salts `salt-a1030`, `salt-b1030`, 2026-10-30):

| task | source | cost | ratio |
|---|---|---:|---:|
| invsqrt | yoyobar, *LB 22.99 - invsqrt 101 - 62 qubits* output (62 qubits, 1327.5 mean Toffolis, sha 67482db6) | 100,693 | 20.625 |
| dqi | 889-qubit decoder in the same yoyobar output (427,475 mean Toffolis, 0 value / 0 phase errors at both salts, sha 73e1b187) | 4.0246e11 | 60.73 |
| dfthcsos | Ahmed Berat Ozer's weights (lambda_eff 48.2956, T 16266, Q 430) on the shared 15 angle circuits, as packaged in manwithacat's *The Real $150K* output (npz sha 47427ee0) | 3.37798e8 | 9.934 |

Projected geometric mean **23.17** (local salts). CPU only, no GPU.
Credits: yoyobar, manwithacat, Ahmed Berat Ozer, heliosli, Dmitrii Gluzdov, Atakan Aldemir, somejoint, busyaprime, mrmeows, alexchilton (all Apache-2.0 public work).

In [ ]:
import glob, hashlib, json, math, os, shutil, sys, zipfile, pathlib
COMP = next(p for p in ['/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation','/kaggle/input/google-verifiable-quantum-circuit-compilation'] if os.path.isdir(p))
def find(name, must):
    hits = [h for h in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if must in h]
    assert hits, (name, must); return hits[0]
YOYO = os.path.dirname(find('submission.zip', 'invsqrt-101'))
MWC = os.path.dirname(find('submission.zip', '150k'))
WORK = pathlib.Path('/kaggle/working'); SUB = WORK/'sub'
shutil.rmtree(SUB, ignore_errors=True); SUB.mkdir()
with zipfile.ZipFile(f'{YOYO}/submission.zip') as z:
    z.extract('invsqrt.kmb', SUB); z.extract('dqi.kmb', SUB)
with zipfile.ZipFile(f'{MWC}/submission.zip') as z:
    for n in z.namelist():
        if n.startswith('dfthcsos/'): z.extract(n, SUB)
EXPECT = {
 'invsqrt.kmb': '67482db6c53896348eadba126924868ed3011d7e1576a6d0abb6f7c137ae4997',
 'dqi.kmb': '73e1b187b288f3cdc7d345214dd78ae6857d2f419041caf02bf3e29d26f7e6f2',
 'dfthcsos/submission.npz': '47427ee032d3719931abeea0ac88f5a6656971eabd492b9d499cae1b42c6b34b',
}
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda: f.read(1<<22), b''): h.update(c)
    return h.hexdigest()
for k,v in EXPECT.items():
    assert sha(SUB/k) == v, k
print('members match pinned hashes;', len(list((SUB/'dfthcsos').glob('angle_k_*.kmb'))), 'angle circuits')

In [ ]:
sys.path[:0] = [f'{COMP}/kickmix', f'{COMP}/verifiers']
import kickmix as km
from invsqrt import verify_submission_invsqrt as vinv
from dqi import verify_submission_dqi as vdqi
from dfthcsos import verify as vdft
SALT = 'v16-commit'
QUICK = os.environ.get('KAGGLE_KERNEL_RUN_TYPE','') != 'Batch'  # interactive: quick check
spec = vinv.ProblemSpecInvSqrt.for_n_p(10)
if QUICK: spec = __import__('dataclasses').replace(spec, shots_per_x=1)
r_inv = vinv.verify(km.Circuit.from_file(str(SUB/'invsqrt.kmb'), format='kmb'), spec, salt=SALT, batch_size=4 if QUICK else 256)
r_dqi = vdqi.verify(km.Circuit.from_file(str(SUB/'dqi.kmb'), format='kmb'), vdqi.ProblemSpecDQI.benchmark_instance(), salt=SALT)
r_ch = vdft.verify(str(SUB/'dfthcsos'), salt=SALT)
REF = {'invsqrt': 102349*20.291, 'dqi': 4.1294e11*59.18, 'dfthcsos': 3.40572e8*9.853}  # calibrated on our scored v14
ratios = []
for t, r in (('invsqrt', r_inv), ('dqi', r_dqi), ('dfthcsos', r_ch)):
    assert r['status'] == 'PASS', (t, r)
    ratios.append(REF[t]/float(r['score'])); print(t, 'PASS', float(r['score']), round(ratios[-1], 4))
print('dqi errors (value/phase):', r_dqi['value_errors'], r_dqi['phase_errors'])
print('projected GM', round(math.prod(ratios)**(1/3), 4))

In [ ]:
with zipfile.ZipFile(WORK/'submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob('*')):
        if p.is_file(): z.write(p, p.relative_to(SUB).as_posix())
shutil.rmtree(SUB); print('wrote submission.zip')